[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camiloandcu/Retail-Demand-Forecasting/blob/main/03_non_neural_baselines.ipynb)

# 03 - Non-neural baselines

This notebook evaluates four pre-registered configurations on the same rolling-origin folds: zero, weekly seasonal naive, a four-week seasonal average, and one global LightGBM model. It writes `artifacts/metrics/baseline_results.csv`, OOF predictions, figures, a champion contract, and local MLflow runs. No recurrent model is trained.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys
from getpass import getpass
from pathlib import Path

REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
BRANCH = "feat-baselines-and-academic-document"  # Change to "main" before merging.
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO_ROOT = Path("/content/Retail-Demand-Forecasting")
    if not (REPO_ROOT / "pyproject.toml").is_file():
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                BRANCH,
                "--single-branch",
                "--depth",
                "1",
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", BRANCH],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(REPO_ROOT), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
        )
else:
    REPO_ROOT = Path.cwd().resolve()
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise RuntimeError("Run the notebook from the repository root.")
os.chdir(REPO_ROOT)
if importlib.util.find_spec("pip") is not None:
    install_command = [sys.executable, "-m", "pip", "install", "-e", ".[notebook,dev,models]"]
elif shutil.which("uv"):
    install_command = [
        "uv",
        "pip",
        "install",
        "--python",
        sys.executable,
        "-e",
        ".[notebook,dev,models]",
    ]
else:
    raise RuntimeError("This environment provides neither pip nor uv.")
subprocess.run(install_command, check=True)
source_path = str(REPO_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
print(f"Repository ready at {REPO_ROOT} with Python {sys.version.split()[0]}")

Repository ready at /content/Retail-Demand-Forecasting with Python 3.13.15


In [2]:
import pandas as pd
from IPython.display import display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.config import load_config
from retail_forecast.data import (
    load_frames,
    required_files_exist,
    validate_dataset,
    write_synthetic_dataset,
)
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.experiments import MODEL_IDS, run_baseline_experiments
from retail_forecast.reproducibility import set_global_seed

MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("RETAIL_FORECAST_MODE must be 'full' or 'smoke'.")
config = load_config(f"configs/{MODE}.yaml")
set_global_seed(config.runtime.seed)
print(f"mode={MODE} | configurations={len(MODEL_IDS)} | tree={config.baseline.tree_model}")

mode=full | configurations=4 | tree=lightgbm


## Data

Existing CSVs and a locally available competition ZIP are reused automatically. If neither is present, the next cell asks for a Kaggle API token using hidden input.

In [3]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir,
        (REPO_ROOT / "data", REPO_ROOT),
        token_reader=getpass,
    )
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
dataset_summary = validate_dataset(config)
frames = load_frames(config.paths.raw_data_dir)
display(dataset_summary.to_dict())

100%|██████████| 21.4M/21.4M [00:00<00:00, 113MB/s] 


Extracting files...


{'train_rows': 3000888,
 'test_rows': 28512,
 'stores': 54,
 'families': 33,
 'series': 1782,
 'train_start': '2013-01-01',
 'train_end': '2017-08-15',
 'test_start': '2017-08-16',
 'test_end': '2017-08-31',
 'horizon': 16,
 'source': 'kaggle'}

## Decisions fixed before training

LightGBM is the only tree model. Its histogram-based CPU training suits the global panel, while integer categorical features can be declared directly without one-hot expansion. Targets are learned on the `log1p` scale and negative raw outputs are clipped to zero before RMSLE.

The experiment budget is four configurations—well below the limit of ten. A recurrent model will only qualify if its global OOF RMSLE is at least 2% below LightGBM on these same folds. Smoke results validate code only and cannot select a champion.

In [6]:
run = run_baseline_experiments(
    config,
    frames,
    REPO_ROOT / "artifacts/metrics",
    REPO_ROOT / "artifacts/figures/baselines",
    REPO_ROOT / "mlruns",
)

FeatureContractError: Target dates must be the contiguous horizon after origin

## Quality and cost

The first table compares global OOF quality with measured fitting cost. The second keeps fold variation visible; the complete CSV also contains results by horizon, store, family, promotion status, and temporal regime.

In [ ]:
display(run["cost_quality"])
display(
    run["results"]
    .loc[run["results"]["scope"].isin(["global", "fold"])]
    .pivot_table(index=["scope", "segment"], columns="model_id", values="rmsle")
)
display(
    run["results"]
    .loc[run["results"]["scope"].eq("horizon")]
    .pivot(index="segment", columns="model_id", values="rmsle")
)

,model_id,duration_seconds,process_max_rss_kib,max_training_rows,rmsle
0,lightgbm_global,0.697961,295936,128,0.816407
1,seasonal_average_4w,0.014945,295808,0,0.819074
2,seasonal_naive_7d,0.014981,295808,0,0.942630
3,zero,0.000006,295808,0,2.438905


model_id        lightgbm_global  seasonal_average_4w  seasonal_naive_7d  \
scope  segment                                                            
fold   fold_1          0.750740             0.808779           0.988948   
       fold_2          0.877171             0.829242           0.893915   
global all             0.816407             0.819074           0.942630   

model_id            zero  
scope  segment            
fold   fold_1   2.356347  
       fold_2   2.518758  
global all      2.438905

model_id,lightgbm_global,seasonal_average_4w,seasonal_naive_7d,zero
segment,,,,
1,0.682150,0.627569,0.811946,2.721689
10,0.926824,0.887149,0.998446,2.252917
11,0.704513,0.650584,0.765190,2.312277
12,0.563647,0.562036,0.701983,2.410300
13,0.567528,0.719622,0.884640,2.470335
14,0.809800,0.883930,1.011693,2.296880
15,0.603631,0.665205,0.277242,2.239571
16,0.464545,0.504681,0.229502,2.343783
2,0.927280,0.959578,1.125937,2.553200


## Handoff

The business baseline is selected only between the two transparent seasonal methods. LightGBM remains the non-neural ML reference even if a sanity or seasonal method happens to score better. The recurrent-model threshold is derived from that fixed reference.

In [ ]:
display(run["champion"])
display(run["manifest"])
for name, path in run["paths"].items():
    print(f"{name}: {path}")

{'mode': 'smoke',
 'empirical_results_are_synthetic': True,
 'business_baseline': 'seasonal_average_4w',
 'business_baseline_rmsle': 0.8190743263066564,
 'non_neural_reference': 'lightgbm_global',
 'non_neural_reference_rmsle': 0.8164066711387608,
 'neural_min_relative_improvement': 0.02,
 'neural_rmsle_threshold': 0.8000785377159855,
 'configuration_count': 4,
 'tree_parameters': {'objective': 'regression',
  'n_estimators': 30,
  'learning_rate': 0.05,
  'num_leaves': 15,
  'min_child_samples': 5,
  'colsample_bytree': 0.8,
  'reg_lambda': 1.0,
  'random_state': 2026,
  'n_jobs': -1,
  'deterministic': True,
  'force_col_wise': True,
  'verbosity': -1},
 'limitations_for_recurrent_model': ['Origin-level summaries do not preserve the full order of the 56-day history.',
  'One pooled tree objective may underrepresent intermittent, high-zero series.',
  'Separate horizon rows cannot learn a joint 16-step sequence representation.']}

{'mode': 'smoke',
 'source': 'synthetic',
 'models': ['zero',
  'seasonal_naive_7d',
  'seasonal_average_4w',
  'lightgbm_global'],
 'tree_selected_before_training': 'lightgbm',
 'folds': [{'name': 'fold_1',
   'origin': '2020-02-01',
   'train_start': '2020-01-01',
   'train_end': '2020-02-01',
   'validation_start': '2020-02-02',
   'validation_end': '2020-02-17',
   'horizon': 16},
  {'name': 'fold_2',
   'origin': '2020-02-17',
   'train_start': '2020-01-01',
   'train_end': '2020-02-17',
   'validation_start': '2020-02-18',
   'validation_end': '2020-03-04',
   'horizon': 16}],
 'training_origins_by_fold': {'fold_1': ['2020-01-16'],
  'fold_2': ['2020-01-16', '2020-02-01']},
 'feature_contract': {'count': 73,
  'numeric': ['onpromotion',
   'horizon',
   'onpromotion_log1p',
   'is_promoted',
   'day_of_week',
   'day_of_month',
   'month',
   'is_weekend',
   'is_month_start',
   'is_month_end',
   'is_payday',
   'day_of_year_sin',
   'day_of_year_cos',
   'sales_origin_lag_1',


results: /home/camilo/projects/eco-forecasting/artifacts/metrics/baseline_results.csv
cost_quality: /home/camilo/projects/eco-forecasting/artifacts/metrics/baseline_cost_quality.csv
champion: /home/camilo/projects/eco-forecasting/artifacts/metrics/baseline_champion.json
manifest: /home/camilo/projects/eco-forecasting/artifacts/metrics/baseline_manifest.json
oof: /home/camilo/projects/eco-forecasting/artifacts/smoke/predictions/baseline_oof.csv.gz


## Why test a recurrent model next?

1. Origin-level summaries do not preserve the full order of the 56-day history.
2. One pooled tree objective may underrepresent intermittent, high-zero series.
3. Separate horizon rows cannot learn a joint 16-step sequence representation.

These are representation limitations, not evidence that a recurrent model will perform better.

In [ ]:
checks = {
    "four configurations only": len(MODEL_IDS) == 4,
    "one tree family only": config.baseline.tree_model == "lightgbm",
    "official result artifact": run["paths"]["results"].is_file(),
    "OOF artifact": run["paths"]["oof"].is_file(),
    "MLflow local tracking": Path(run["manifest"]["mlflow_database_path"]).is_file(),
    "neural threshold pre-registered": config.baseline.neural_min_relative_improvement == 0.02,
}
gate = pd.DataFrame(
    [{"check": name, "status": "PASS" if passed else "FAIL"} for name, passed in checks.items()]
)
display(gate)
if not all(checks.values()):
    raise RuntimeError("Baseline gate failed.")
print(f"G4 {MODE.upper()} PASS")

,check,status
0,four configurations only,PASS
1,one tree family only,PASS
2,official result artifact,PASS
3,OOF artifact,PASS
4,MLflow local tracking,PASS
5,neural threshold pre-registered,PASS


G4 SMOKE PASS
